# Pipeline A/B Comparison: Full vs. Simple Retrieval

**Question**: Does query expansion + hybrid BM25/dense search justify its cost over dense-only retrieval?

| | Full Pipeline | Simple Pipeline |
|---|---|---|
| Query expansion | Claude generates 3 alternatives | None |
| Retrieval | BM25 + Pinecone dense, RRF fusion | Pinecone dense only |
| LLM calls/query | 2 (expand + generate) | 1 (generate only) |
| Pinecone calls/query | 4 (one per expanded query) | 1 |
| Reranking | Cross-encoder (same) | Cross-encoder (same) |

**Important**: This is a cost-vs-quality tradeoff comparison, not a controlled ablation study.
The two pipelines differ in multiple dimensions (expansion, BM25, candidate pool size),
so quality differences cannot be attributed to any single factor.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

## 1. Initialize Pipelines

In [ ]:
from src.embeddings.vector_store import VectorStore
from src.generation.llm_client import LLMClient
from src.generation.rag_chain import RAGPipeline
from src.retrieval.hybrid_search import HybridSearcher
from src.retrieval.reranker import Reranker
from src.retrieval.strategies import (
    DenseRetrievalStrategy,
    HybridRetrievalStrategy,
)

store = VectorStore()
searcher = HybridSearcher(vector_store=store)
reranker = Reranker()
llm = LLMClient()

# Share parquet data between strategies
chunk_lookup = {m['chunk_id']: m for m in searcher.chunk_metadata}

pipelines = {
    'full': RAGPipeline(
        strategy=HybridRetrievalStrategy(searcher=searcher, llm_client=llm),
        reranker=reranker, llm_client=llm,
    ),
    'simple': RAGPipeline(
        strategy=DenseRetrievalStrategy(vector_store=store, chunk_lookup=chunk_lookup),
        reranker=reranker, llm_client=llm,
    ),
}
print(f'Pipelines initialized: {list(pipelines.keys())}')

## 2. Run Evaluation

In [ ]:
from src.evaluation.eval_harness import PipelineEvaluator, compare_pipelines

# Quick preview with 20 queries (change to None for full 500)
N_QUERIES = 20

comparison = compare_pipelines(pipelines, n_queries=N_QUERIES)
comparison

## 3. Load Per-Query Results

In [ ]:
full_df = pd.read_csv('../data/exports/eval_full_results.csv')
simple_df = pd.read_csv('../data/exports/eval_simple_results.csv')

print(f'Full pipeline: {len(full_df)} queries')
print(f'Simple pipeline: {len(simple_df)} queries')

## 4. Headline Metrics Comparison

In [ ]:
metrics_display = comparison[[
    'precision_at_5', 'mrr', 'answer_token_overlap',
    'guardrail_pass_rate', 'mean_latency_s', 'p95_latency_s',
    'total_llm_calls', 'total_pinecone_calls',
]].T

if 'full' in metrics_display.columns and 'simple' in metrics_display.columns:
    metrics_display['delta_%'] = (
        (metrics_display['simple'] - metrics_display['full'])
        / metrics_display['full'].replace(0, np.nan) * 100
    ).round(1)

metrics_display

## 5. Quality Metrics Bar Chart

In [ ]:
quality_metrics = ['precision_at_5', 'mrr', 'answer_token_overlap', 'guardrail_pass_rate']
x = np.arange(len(quality_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
if 'full' in comparison.index and 'simple' in comparison.index:
    bars1 = ax.bar(x - width/2, [comparison.loc['full', m] for m in quality_metrics], width, label='Full', color='#2196F3')
    bars2 = ax.bar(x + width/2, [comparison.loc['simple', m] for m in quality_metrics], width, label='Simple', color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('Quality Metrics: Full vs. Simple Pipeline')
ax.set_xticks(x)
ax.set_xticklabels(['Precision@5', 'MRR', 'Answer Overlap', 'Guardrail Pass'])
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../data/exports/ab_quality_comparison.png', dpi=150)
plt.show()

## 6. Latency Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(full_df['total_latency_s'], bins=20, alpha=0.6, label='Full', color='#2196F3')
ax.hist(simple_df['total_latency_s'], bins=20, alpha=0.6, label='Simple', color='#FF9800')
ax.axvline(full_df['total_latency_s'].median(), color='#1565C0', linestyle='--', label=f'Full median: {full_df["total_latency_s"].median():.1f}s')
ax.axvline(simple_df['total_latency_s'].median(), color='#E65100', linestyle='--', label=f'Simple median: {simple_df["total_latency_s"].median():.1f}s')
ax.set_xlabel('Latency (seconds)')
ax.set_ylabel('Query count')
ax.set_title('Per-Query Latency Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../data/exports/ab_latency_distribution.png', dpi=150)
plt.show()

## 7. Per-Specialty Breakdown

In [ ]:
full_by_qtype = full_df.groupby('qtype')['precision_at_k'].mean()
simple_by_qtype = simple_df.groupby('qtype')['precision_at_k'].mean()

qtype_comparison = pd.DataFrame({
    'full': full_by_qtype,
    'simple': simple_by_qtype,
}).dropna()
qtype_comparison['delta'] = qtype_comparison['full'] - qtype_comparison['simple']
qtype_comparison = qtype_comparison.sort_values('delta', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(qtype_comparison))
width = 0.35
ax.bar(x - width/2, qtype_comparison['full'], width, label='Full', color='#2196F3')
ax.bar(x + width/2, qtype_comparison['simple'], width, label='Simple', color='#FF9800')
ax.set_ylabel('Precision@5')
ax.set_title('Precision@5 by Medical Specialty')
ax.set_xticks(x)
ax.set_xticklabels(qtype_comparison.index, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('../data/exports/ab_per_qtype.png', dpi=150)
plt.show()

print('\nSpecialties where full pipeline wins most:')
print(qtype_comparison.head())

## 8. Cost Analysis

In [ ]:
# Cost model for Claude Haiku 4.5 and Pinecone
HAIKU_INPUT_COST_PER_M = 0.80   # $/M input tokens
HAIKU_OUTPUT_COST_PER_M = 4.00  # $/M output tokens
AVG_INPUT_TOKENS = 1000
AVG_OUTPUT_TOKENS = 500

def estimate_cost_per_query(llm_calls, pinecone_calls):
    llm_cost = llm_calls * (
        AVG_INPUT_TOKENS / 1e6 * HAIKU_INPUT_COST_PER_M
        + AVG_OUTPUT_TOKENS / 1e6 * HAIKU_OUTPUT_COST_PER_M
    )
    return llm_cost  # Pinecone cost is negligible per query

full_cost = estimate_cost_per_query(llm_calls=2, pinecone_calls=4)
simple_cost = estimate_cost_per_query(llm_calls=1, pinecone_calls=1)

# Scale to daily usage: 200 students, 10 queries/day each
daily_queries = 200 * 10
monthly_queries = daily_queries * 22  # Weekdays

cost_table = pd.DataFrame({
    'Full': [full_cost, full_cost * daily_queries, full_cost * monthly_queries],
    'Simple': [simple_cost, simple_cost * daily_queries, simple_cost * monthly_queries],
    'Savings': [
        full_cost - simple_cost,
        (full_cost - simple_cost) * daily_queries,
        (full_cost - simple_cost) * monthly_queries,
    ],
}, index=['Per query', 'Daily (2K queries)', 'Monthly (44K queries)'])

cost_table = cost_table.applymap(lambda x: f'${x:.4f}' if x < 1 else f'${x:.2f}')
cost_table

## 9. Conclusion

### Tradeoff Summary

This comparison measures the **total cost-vs-quality tradeoff** between the full and simple pipelines.
The full pipeline combines three advantages: query expansion (catching synonym mismatches),
BM25 keyword retrieval (complementing dense embeddings), and a larger candidate pool for the reranker.

For a medical school environment with interactive tutoring sessions where response time matters,
the simple pipeline may be preferable if the quality degradation is marginal.
For exam preparation or clinical reference scenarios where accuracy is paramount,
the full pipeline's additional cost is justified.

### Architecture Decision

The Strategy pattern allows switching between modes at runtime via the API's `mode` parameter,
so this doesn't have to be an either/or decision — different use cases can use different modes.